In [ ]:
!pip -q install kagglehub

In [ ]:
import os
import json
from collections import Counter

import kagglehub
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
import os
import json
from typing import List

import kagglehub
import pandas as pd

# -------------------------
# Config
# -------------------------
TARGET_CITIES = ["Philadelphia"]   # add more later if needed
KEEP_ONLY_OPEN = True
OUTPUT_DIR = "/content/yelp_philly_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download dataset
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
print("Dataset path:", path)

BUSINESS_FILE = os.path.join(path, "yelp_academic_dataset_business.json")
REVIEW_FILE   = os.path.join(path, "yelp_academic_dataset_review.json")
USER_FILE     = os.path.join(path, "yelp_academic_dataset_user.json")

100%|██████████| 4.07G/4.07G [03:02<00:00, 24.0MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4


In [ ]:
def normalize_city(city: str) -> str:
    return str(city).strip().lower()


TARGET_CITY_SET = {normalize_city(city) for city in TARGET_CITIES}


def is_restaurant_business(categories: str) -> bool:
    if not categories:
        return False
    category_list = [c.strip() for c in str(categories).split(",")]
    return "Restaurants" in category_list


def safe_json_dumps(x):
    if x is None:
        return None
    if isinstance(x, (dict, list)):
        return json.dumps(x)
    return str(x)

In [ ]:
def preprocess_businesses(filepath: str) -> pd.DataFrame:
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)

            city = normalize_city(row.get("city", ""))
            if city not in TARGET_CITY_SET:
                continue

            if KEEP_ONLY_OPEN and row.get("is_open", 0) != 1:
                continue

            if not row.get("business_id") or not row.get("name"):
                continue

            categories = row.get("categories")
            if not is_restaurant_business(categories):
                continue

            rows.append({
                "business_id": row["business_id"],
                "name": row["name"],
                "address": row.get("address"),
                "city": row.get("city"),
                "state": row.get("state"),
                "postal_code": row.get("postal_code"),
                "latitude": row.get("latitude"),
                "longitude": row.get("longitude"),
                "stars": row.get("stars"),
                "review_count": row.get("review_count"),
                "is_open": row.get("is_open"),
                "categories": row.get("categories"),         # keep raw for later
                "attributes_json": safe_json_dumps(row.get("attributes")),
                "hours_json": safe_json_dumps(row.get("hours"))
            })

    df = pd.DataFrame(rows)
    return df

In [ ]:
def preprocess_reviews(filepath: str, valid_business_ids: set) -> pd.DataFrame:
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            row = json.loads(line)

            if row.get("business_id") not in valid_business_ids:
                continue

            if not row.get("review_id") or not row.get("user_id") or not row.get("text"):
                continue

            dt = pd.to_datetime(row.get("date"), errors="coerce")
            if pd.isna(dt):
                continue

            rows.append({
                "review_id": row["review_id"],
                "user_id": row["user_id"],
                "business_id": row["business_id"],
                "stars": row.get("stars"),
                "useful": row.get("useful"),
                "funny": row.get("funny"),
                "cool": row.get("cool"),
                "text": row.get("text"),
                "date": dt.strftime("%Y-%m-%d"),
                "review_year": dt.year,
                "review_month": dt.month
            })

            if (i + 1) % 500000 == 0:
                print(f"Processed {i+1:,} review lines... kept {len(rows):,}")

    df = pd.DataFrame(rows)
    return df

In [ ]:
def preprocess_users(filepath: str, valid_user_ids: set) -> pd.DataFrame:
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)

            if row.get("user_id") not in valid_user_ids:
                continue

            if not row.get("user_id"):
                continue

            yelping_since = pd.to_datetime(row.get("yelping_since"), errors="coerce")
            yelping_year = None if pd.isna(yelping_since) else yelping_since.year

            rows.append({
                "user_id": row["user_id"],
                "name": row.get("name"),
                "review_count": row.get("review_count"),
                "average_stars": row.get("average_stars"),
                "fans": row.get("fans"),
                "useful": row.get("useful"),
                "funny": row.get("funny"),
                "cool": row.get("cool"),
                "yelping_since": None if pd.isna(yelping_since) else yelping_since.strftime("%Y-%m-%d"),
                "yelping_year": yelping_year,
                "elite": safe_json_dumps(row.get("elite")),
                "friends": safe_json_dumps(row.get("friends"))
            })

    df = pd.DataFrame(rows)
    return df

In [ ]:
print("Processing businesses...")
businesses = preprocess_businesses(BUSINESS_FILE)
valid_business_ids = set(businesses["business_id"])
print("Businesses kept:", len(businesses))

print("\nProcessing reviews...")
reviews = preprocess_reviews(REVIEW_FILE, valid_business_ids)
valid_user_ids = set(reviews["user_id"])
print("Reviews kept:", len(reviews))

print("\nProcessing users...")
users = preprocess_users(USER_FILE, valid_user_ids)
print("Users kept:", len(users))

Processing businesses...
Businesses kept: 3527

Processing reviews...
Reviews kept: 511311

Processing users...
Users kept: 178369


In [ ]:
businesses.to_csv(os.path.join(OUTPUT_DIR, "businesses.csv"), index=False)
reviews.to_csv(os.path.join(OUTPUT_DIR, "reviews.csv"), index=False)
users.to_csv(os.path.join(OUTPUT_DIR, "users.csv"), index=False)

print("\nSaved files:")
print("-", os.path.join(OUTPUT_DIR, "businesses.csv"))
print("-", os.path.join(OUTPUT_DIR, "reviews.csv"))
print("-", os.path.join(OUTPUT_DIR, "users.csv"))


Saved files:
- /content/yelp_philly_processed/businesses.csv
- /content/yelp_philly_processed/reviews.csv
- /content/yelp_philly_processed/users.csv


In [ ]:
print("businesses shape:", businesses.shape)
print("reviews shape:", reviews.shape)
print("users shape:", users.shape)

print("\nUnique cities kept:", businesses["city"].nunique())
print("Restaurants with at least 1 kept review:", reviews["business_id"].nunique())
print("Unique users with kept reviews:", reviews["user_id"].nunique())

display(businesses.head())
display(reviews.head())
display(users.head())

businesses shape: (3527, 14)
reviews shape: (511311, 11)
users shape: (178369, 12)

Unique cities kept: 3
Restaurants with at least 1 kept review: 3527
Unique users with kept reviews: 178369


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories,attributes_json,hours_json
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{""RestaurantsDelivery"": ""False"", ""OutdoorSeati...","{""Monday"": ""7:0-20:0"", ""Tuesday"": ""7:0-20:0"", ..."
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,1,"Sushi Bars, Restaurants, Japanese","{""RestaurantsReservations"": ""True"", ""Restauran...","{""Tuesday"": ""13:30-22:0"", ""Wednesday"": ""13:30-..."
2,ROeacJQwBeh05Rqg7F6TCg,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,1,"Korean, Restaurants","{""NoiseLevel"": ""u'quiet'"", ""GoodForMeal"": ""{'d...","{""Monday"": ""11:30-20:30"", ""Tuesday"": ""11:30-20..."
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,901 N Delaware Ave,Philadelphia,PA,19123,39.962582,-75.135657,3.5,65,1,"Eatertainment, Arts & Entertainment, Brewpubs,...","{""OutdoorSeating"": ""True"", ""RestaurantsPriceRa...","{""Monday"": ""0:0-0:0"", ""Wednesday"": ""16:0-22:0""..."
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,3604 Chestnut St,Philadelphia,PA,19104,39.954573,-75.194894,3.0,56,1,"Restaurants, Automotive, Delis, Gas Stations, ...","{""Alcohol"": ""u'none'"", ""RestaurantsGoodForGrou...","{""Monday"": ""0:0-0:0"", ""Tuesday"": ""0:0-0:0"", ""W..."


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,review_year,review_month
0,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04,2015,1
1,oyaMhzBSwfGgemSGuZCdwQ,Dd1jQj7S-BFGqRbApFzCFw,YtSqYv1Q_pOltsVPSx54SA,5.0,0,0,0,Tremendous service (Big shout out to Douglas) ...,2013-06-24,2013,6
2,Xs8Z8lmKkosqW5mw_sVAoA,IQsF3Rc6IgCzjVV9DE8KXg,eFvzHawVJofxSnD7TgbZtg,5.0,0,0,0,My absolute favorite cafe in the city. Their b...,2014-11-12,2014,11
3,JBWZmBy69VMggxj3eYn17Q,aFa96pz67TwOFu4Weq5Agg,kq5Ghhh14r-eCxlVmlyd8w,5.0,0,0,0,My boyfriend and I tried this deli for the fir...,2018-08-23,2018,8
4,YcLXh-3UC9y6YFAI9xxzPQ,G0DHgkSsDozqUPWtlxVEMw,oBhJuukGRqPVvYBfTkhuZA,4.0,0,0,0,The only reason I didn't give this restaurant ...,2015-03-05,2015,3


,user_id,name,review_count,average_stars,fans,useful,funny,cool,yelping_since,yelping_year,elite,friends
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,3.91,267,7217,1259,5994,2007-01-25,2007,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA..."
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,3.74,3138,43091,13066,27281,2009-01-25,2009,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A..."
2,q_QQ5kBBwlCcbL1s4NVK3g,Jane,1221,3.85,1357,14953,9940,11211,2005-03-14,2005,"2006,2007,2008,2009,2010,2011,2012,2013,2014","xBDpTUbai0DXrvxCe3X16Q, 7GPNBO496aecrjJfW6UWtg..."
3,AUi8MPWJ0mLkMfwbui27lg,John,109,3.40,4,154,20,23,2010-01-07,2010,,"gy5fWeSv3Gamuq9Ox4MV4g, lMr3LWU6kPFLTmCpDkACxg..."
4,1McG5Rn_UDkmlkZOrsdptg,Teresa,7,4.29,1,18,3,13,2009-05-26,2009,,"piejMEdRkGB7-1aL4lL5NQ, X0zFOU6iG95-feQKOXkgrA..."


In [ ]:
# =========================================================
# Config
# =========================================================
DATA_DIR = "/content/yelp_philly_processed"

BUSINESSES_PATH = os.path.join(DATA_DIR, "businesses.csv")
REVIEWS_PATH = os.path.join(DATA_DIR, "reviews.csv")
USERS_PATH = os.path.join(DATA_DIR, "users.csv")

OUTPUT_DIR = os.path.join(DATA_DIR, "splits_temporal")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# temporal split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# define cold-start from TRAIN only
COLD_START_MAX_TRAIN_REVIEWS = 5

# main benchmark:
# False = keep only items seen at least once in train for val/test
# True  = allow zero-train items in val/test
ALLOW_ZERO_TRAIN_ITEMS_IN_EVAL = False

# optional user-history filter
# set to 0 to disable
MIN_TRAIN_USER_INTERACTIONS = 1

# optional save of summary csvs
SAVE_SUMMARY_TABLES = True


# =========================================================
# Load processed files
# =========================================================
businesses = pd.read_csv(BUSINESSES_PATH)
reviews = pd.read_csv(REVIEWS_PATH)
users = pd.read_csv(USERS_PATH)

print("Loaded:")
print("businesses:", businesses.shape)
print("reviews:", reviews.shape)
print("users:", users.shape)


# =========================================================
# Clean and sort reviews by time
# =========================================================
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")
reviews = reviews.dropna(subset=["date", "user_id", "business_id"]).copy()
reviews = reviews.sort_values("date").reset_index(drop=True)

print("\nReviews after cleanup:", reviews.shape)
print("Date range:", reviews["date"].min(), "to", reviews["date"].max())


# =========================================================
# Global temporal split
# =========================================================
train_cutoff = reviews["date"].quantile(TRAIN_RATIO)
val_cutoff = reviews["date"].quantile(TRAIN_RATIO + VAL_RATIO)

train_reviews = reviews[reviews["date"] <= train_cutoff].copy()
val_reviews = reviews[(reviews["date"] > train_cutoff) & (reviews["date"] <= val_cutoff)].copy()
test_reviews = reviews[reviews["date"] > val_cutoff].copy()

print("\nInitial time split:")
print("train:", train_reviews.shape)
print("val:", val_reviews.shape)
print("test:", test_reviews.shape)
print("train cutoff:", train_cutoff)
print("val cutoff:", val_cutoff)


# =========================================================
# Keep only users seen in train
# =========================================================
train_user_ids = set(train_reviews["user_id"].unique())

val_reviews = val_reviews[val_reviews["user_id"].isin(train_user_ids)].copy()
test_reviews = test_reviews[test_reviews["user_id"].isin(train_user_ids)].copy()

print("\nAfter keeping only users seen in train:")
print("train:", train_reviews.shape)
print("val:", val_reviews.shape)
print("test:", test_reviews.shape)


# =========================================================
# Optional: require minimum train interactions per user
# =========================================================
if MIN_TRAIN_USER_INTERACTIONS > 1:
    train_user_counts = (
        train_reviews.groupby("user_id")
        .size()
        .reset_index(name="train_user_review_count")
    )

    eligible_train_users = set(
        train_user_counts.loc[
            train_user_counts["train_user_review_count"] >= MIN_TRAIN_USER_INTERACTIONS,
            "user_id"
        ]
    )

    train_reviews = train_reviews[train_reviews["user_id"].isin(eligible_train_users)].copy()
    val_reviews = val_reviews[val_reviews["user_id"].isin(eligible_train_users)].copy()
    test_reviews = test_reviews[test_reviews["user_id"].isin(eligible_train_users)].copy()

    print(f"\nAfter enforcing MIN_TRAIN_USER_INTERACTIONS >= {MIN_TRAIN_USER_INTERACTIONS}:")
    print("train:", train_reviews.shape)
    print("val:", val_reviews.shape)
    print("test:", test_reviews.shape)


# =========================================================
# Compute TRAIN restaurant counts
# =========================================================
train_business_counts = (
    train_reviews.groupby("business_id")
    .size()
    .reset_index(name="train_review_count")
)

businesses_with_train_info = businesses.merge(
    train_business_counts,
    on="business_id",
    how="left"
)

businesses_with_train_info["train_review_count"] = (
    businesses_with_train_info["train_review_count"]
    .fillna(0)
    .astype(int)
)

# cold-start = 1..threshold reviews in train
businesses_with_train_info["cold_start_in_train"] = (
    (businesses_with_train_info["train_review_count"] >= 1) &
    (businesses_with_train_info["train_review_count"] <= COLD_START_MAX_TRAIN_REVIEWS)
)

# zero-train bucket
businesses_with_train_info["zero_train_reviews"] = (
    businesses_with_train_info["train_review_count"] == 0
)

seen_train_business_ids = set(
    businesses_with_train_info.loc[
        businesses_with_train_info["train_review_count"] >= 1,
        "business_id"
    ]
)

cold_start_business_ids = set(
    businesses_with_train_info.loc[
        businesses_with_train_info["cold_start_in_train"],
        "business_id"
    ]
)

zero_train_business_ids = set(
    businesses_with_train_info.loc[
        businesses_with_train_info["zero_train_reviews"],
        "business_id"
    ]
)

print("\nTrain business summary:")
print("businesses seen in train:", len(seen_train_business_ids))
print(f"cold-start businesses (1 to {COLD_START_MAX_TRAIN_REVIEWS} train reviews):", len(cold_start_business_ids))
print("zero-train businesses:", len(zero_train_business_ids))


# =========================================================
# Main benchmark option: remove zero-train items from val/test
# =========================================================
if not ALLOW_ZERO_TRAIN_ITEMS_IN_EVAL:
    val_reviews = val_reviews[val_reviews["business_id"].isin(seen_train_business_ids)].copy()
    test_reviews = test_reviews[test_reviews["business_id"].isin(seen_train_business_ids)].copy()

    print("\nAfter dropping zero-train items from val/test:")
    print("train:", train_reviews.shape)
    print("val:", val_reviews.shape)
    print("test:", test_reviews.shape)


# =========================================================
# Add cold-start labels to each split
# =========================================================
train_reviews["cold_start_in_train"] = train_reviews["business_id"].isin(cold_start_business_ids)
val_reviews["cold_start_in_train"] = val_reviews["business_id"].isin(cold_start_business_ids)
test_reviews["cold_start_in_train"] = test_reviews["business_id"].isin(cold_start_business_ids)

train_reviews["zero_train_reviews"] = False
val_reviews["zero_train_reviews"] = val_reviews["business_id"].isin(zero_train_business_ids)
test_reviews["zero_train_reviews"] = test_reviews["business_id"].isin(zero_train_business_ids)


# =========================================================
# Helper tables
# =========================================================
train_users = pd.DataFrame({"user_id": sorted(train_reviews["user_id"].unique())})
train_items = pd.DataFrame({"business_id": sorted(train_reviews["business_id"].unique())})

cold_start_items = businesses_with_train_info[
    businesses_with_train_info["cold_start_in_train"]
].copy()

warm_items = businesses_with_train_info[
    businesses_with_train_info["train_review_count"] > COLD_START_MAX_TRAIN_REVIEWS
].copy()

zero_train_items = businesses_with_train_info[
    businesses_with_train_info["zero_train_reviews"]
].copy()


# =========================================================
# Summary helpers
# =========================================================
def summarize_split(name, df):
    return {
        "split": name,
        "rows": len(df),
        "unique_users": df["user_id"].nunique(),
        "unique_businesses": df["business_id"].nunique(),
        "date_min": df["date"].min(),
        "date_max": df["date"].max(),
        "cold_start_rows": int(df["cold_start_in_train"].sum()),
        "cold_start_businesses": df.loc[df["cold_start_in_train"], "business_id"].nunique(),
        "zero_train_rows": int(df["zero_train_reviews"].sum()),
    }


train_summary = summarize_split("train", train_reviews)
val_summary = summarize_split("val", val_reviews)
test_summary = summarize_split("test", test_reviews)

summary_df = pd.DataFrame([train_summary, val_summary, test_summary])

print("\nSplit summary:")
print(summary_df)


# =========================================================
# Business summary table
# =========================================================
business_summary_df = pd.DataFrame({
    "metric": [
        "total_businesses",
        "businesses_seen_in_train",
        f"cold_start_businesses_1_to_{COLD_START_MAX_TRAIN_REVIEWS}",
        "zero_train_businesses",
        "median_train_review_count_seen_items",
        "mean_train_review_count_seen_items"
    ],
    "value": [
        len(businesses_with_train_info),
        len(seen_train_business_ids),
        len(cold_start_business_ids),
        len(zero_train_business_ids),
        businesses_with_train_info.loc[
            businesses_with_train_info["train_review_count"] >= 1,
            "train_review_count"
        ].median(),
        businesses_with_train_info.loc[
            businesses_with_train_info["train_review_count"] >= 1,
            "train_review_count"
        ].mean(),
    ]
})

print("\nBusiness summary:")
print(business_summary_df)


# =========================================================
# Save outputs
# =========================================================
train_reviews.to_csv(os.path.join(OUTPUT_DIR, "train_reviews.csv"), index=False)
val_reviews.to_csv(os.path.join(OUTPUT_DIR, "val_reviews.csv"), index=False)
test_reviews.to_csv(os.path.join(OUTPUT_DIR, "test_reviews.csv"), index=False)

businesses_with_train_info.to_csv(
    os.path.join(OUTPUT_DIR, "businesses_with_train_info.csv"),
    index=False
)

train_users.to_csv(os.path.join(OUTPUT_DIR, "train_users.csv"), index=False)
train_items.to_csv(os.path.join(OUTPUT_DIR, "train_items.csv"), index=False)
cold_start_items.to_csv(os.path.join(OUTPUT_DIR, "cold_start_items.csv"), index=False)
warm_items.to_csv(os.path.join(OUTPUT_DIR, "warm_items.csv"), index=False)
zero_train_items.to_csv(os.path.join(OUTPUT_DIR, "zero_train_items.csv"), index=False)

if SAVE_SUMMARY_TABLES:
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "split_summary.csv"), index=False)
    business_summary_df.to_csv(os.path.join(OUTPUT_DIR, "business_summary.csv"), index=False)

print("\nSaved files:")
for fname in [
    "train_reviews.csv",
    "val_reviews.csv",
    "test_reviews.csv",
    "businesses_with_train_info.csv",
    "train_users.csv",
    "train_items.csv",
    "cold_start_items.csv",
    "warm_items.csv",
    "zero_train_items.csv",
    "split_summary.csv",
    "business_summary.csv",
]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        print("-", fpath)


# =========================================================
# Quick previews
# =========================================================
print("\nTrain preview:")
display(train_reviews.head())

print("\nVal preview:")
display(val_reviews.head())

print("\nTest preview:")
display(test_reviews.head())

print("\nBusiness preview:")
display(
    businesses_with_train_info[
        ["business_id", "name", "review_count", "train_review_count", "cold_start_in_train", "zero_train_reviews"]
    ].head()
)

print("\nSplit summary preview:")
display(summary_df)

print("\nBusiness summary preview:")
display(business_summary_df)

Loaded:
businesses: (3527, 14)
reviews: (511311, 11)
users: (178369, 12)

Reviews after cleanup: (511311, 11)
Date range: 2005-02-16 00:00:00 to 2022-01-19 00:00:00

Initial time split:
train: (357928, 11)
val: (76776, 11)
test: (76607, 11)
train cutoff: 2018-08-24 00:00:00
val cutoff: 2019-11-22 00:00:00

After keeping only users seen in train:
train: (357928, 11)
val: (32386, 11)
test: (24932, 11)

Train business summary:
businesses seen in train: 2890
cold-start businesses (1 to 5 train reviews): 327
zero-train businesses: 637

After dropping zero-train items from val/test:
train: (357928, 11)
val: (27914, 11)
test: (17858, 11)

Split summary:
   split    rows  unique_users  unique_businesses   date_min   date_max  \
0  train  357928        124626               2890 2005-02-16 2018-08-24   
1    val   27914          9900               2404 2018-08-25 2019-11-22   
2   test   17858          6647               2225 2019-11-23 2022-01-19   

   cold_start_rows  cold_start_businesses  z

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,review_year,review_month,cold_start_in_train,zero_train_reviews
0,g80vzN72iU03Wh0fSpq41g,3zBJUlWtPNoZ0uN83ODbyg,PP3BBaVxZLcJU54uP_wL6Q,5.0,0,0,0,These guys really are the king of cheese steak...,2005-02-16,2005,2,False,False
1,DTrvaOwqev-xhbqqblt7Tw,H4JNrBAoyCk_ZMZWbAf8OA,Co3Ogqy6y2JgZdG0wBlrUQ,5.0,0,0,4,THIS IS MY FAVORITE BAR IN PHILADELPHIA. Oh T...,2005-05-25,2005,5,False,False
2,ZWmGW-vXMfFzXre4EmwTDw,c8qFkI_VusWo0xZvkjfBWQ,6_T2xzR74JqGCTPefAD8Tw,5.0,0,0,0,Morimoto is one of the coolest looking high en...,2005-05-26,2005,5,False,False
3,VT5V1Exe7LfzsnJ2XjMAPQ,c8qFkI_VusWo0xZvkjfBWQ,IkY2ticzHEn4QFn8hQLSWg,5.0,2,1,1,"Geno's is superb, arguably the best in town. ...",2005-05-26,2005,5,False,False
4,J3Jk5A1TnFeKf2SGNrZLrw,c8qFkI_VusWo0xZvkjfBWQ,RQAF6a0akMiot5lZZnMNNw,5.0,2,1,1,I lost a bet during undergrad at Carnegie Mell...,2005-05-26,2005,5,False,False



Val preview:


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,review_year,review_month,cold_start_in_train,zero_train_reviews
357929,eejBHlLNfV1s_FauXBlOoA,XKMThtOHNk-dJ-ExXFaq6w,C_EtrXTygRX5RTUOKtO6Dg,2.0,1,0,0,I wouldn't go back... nah!\n\nThe only reason ...,2018-08-25,2018,8,False,False
357935,VkvhqUOdoW52IYS9hFIBWw,IB_sYwoVgvgRVw6_G47Gfg,CiTYWOKcXTZYztsT43wb5g,4.0,1,0,0,This is one of our favorite places to go in th...,2018-08-25,2018,8,False,False
357937,Fr_miatA6vkubPrwSpkX5g,CIFrgKgUrr6xyigd1AHeUw,rKggQN5KT6AtOlmuJ8NAZQ,5.0,0,0,1,[UPDATE 8/25/2018] Everything I've previously ...,2018-08-25,2018,8,False,False
357940,IIFKlyarOL2-VS9CHl6EYQ,3z_lfmfHSbkgmq2RRNeedQ,jEasa4Sbzy4NdLyzPPgQyg,2.0,1,0,0,As far as brew pubs go 2nd Story Brewing is ve...,2018-08-25,2018,8,False,False
357946,4X-U6oZhn874rIx1iA2jWw,EqgK3T7NEXy2Uly8YXwz5A,gvD09Ev1aOmphtlq07zYEA,5.0,0,0,0,Carnitas tacos and the chorizo nachos were fan...,2018-08-25,2018,8,False,False



Test preview:


...

,business_id,name,review_count,train_review_count,cold_start_in_train,zero_train_reviews
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,80,63,False,False
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,245,114,False,False
2,ROeacJQwBeh05Rqg7F6TCg,BAP,205,163,False,False
3,aPNXGTDkf-4bjhyMBQxqpQ,Craft Hall,65,0,False,True
4,ppFCk9aQkM338Rgwpl2F5A,Wawa,56,45,False,False



Split summary preview:


,split,rows,unique_users,unique_businesses,date_min,date_max,cold_start_rows,cold_start_businesses,zero_train_rows
0,train,357928,124626,2890,2005-02-16,2018-08-24,1147,327,0
1,val,27914,9900,2404,2018-08-25,2019-11-22,634,193,0
2,test,17858,6647,2225,2019-11-23,2022-01-19,381,153,0



Business summary preview:


,metric,value
0,total_businesses,3527.000000
1,businesses_seen_in_train,2890.000000
2,cold_start_businesses_1_to_5,327.000000
3,zero_train_businesses,637.000000
4,median_train_review_count_seen_items,38.000000
5,mean_train_review_count_seen_items,123.850519


In [ ]:
# =========================================================
# Shared evaluator for all recommendation models
# =========================================================
import math
import numpy as np

MIN_POSITIVE_STARS = 4.0
EVAL_KS = [5, 10, 20]

# If this cell is run later by itself, reload the split files.
if "train_reviews" not in globals():
    SPLIT_DIR = os.path.join(DATA_DIR, "splits_temporal")
    train_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "train_reviews.csv"))
    val_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "val_reviews.csv"))
    test_reviews = pd.read_csv(os.path.join(SPLIT_DIR, "test_reviews.csv"))


def make_seen_map(train_df):
    seen = {}
    for uid, grp in train_df.groupby("user_id"):
        seen[uid] = set(grp["business_id"])
    return seen


def make_truth_map(split_df, min_stars=MIN_POSITIVE_STARS, cold_only=False):
    part = split_df[split_df["stars"] >= min_stars].copy()

    if cold_only:
        part = part[part["cold_start_in_train"] == True].copy()

    truth = {}
    for uid, grp in part.groupby("user_id"):
        truth[uid] = set(grp["business_id"])
    return truth


def recommendation_df_to_map(rec_df, topk=20):
    """Accept either rank_1...rank_k format or user_id/business_id/score format."""
    rank_cols = [c for c in rec_df.columns if c.startswith("rank_")]

    rec_map = {}

    if rank_cols:
        rank_cols = sorted(rank_cols, key=lambda x: int(x.split("_")[1]))
        for row in rec_df[["user_id"] + rank_cols].itertuples(index=False):
            uid = row[0]
            rec_map[uid] = [x for x in row[1:] if pd.notna(x)][:topk]
        return rec_map

    if "score" in rec_df.columns:
        rec_df = rec_df.sort_values(["user_id", "score"], ascending=[True, False])

    for uid, grp in rec_df.groupby("user_id"):
        rec_map[uid] = list(grp["business_id"].head(topk))

    return rec_map


def clean_rec_list(items, seen_items=None, k=10):
    seen_items = seen_items or set()
    out = []
    used = set()

    for item in items:
        if item in used:
            continue
        if item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == k:
            break

    return out


def evaluate_rec_map(rec_map, truth_map, seen_map=None, k=10):
    rows = []

    for uid, true_items in truth_map.items():
        if len(true_items) == 0:
            continue

        seen_items = set() if seen_map is None else seen_map.get(uid, set())
        recs = clean_rec_list(rec_map.get(uid, []), seen_items=seen_items, k=k)

        hit_flags = [1 if item in true_items else 0 for item in recs]
        hits = sum(hit_flags)

        hit_at_k = 1 if hits > 0 else 0
        recall_at_k = hits / len(true_items)

        rr = 0.0
        for rank, hit in enumerate(hit_flags, start=1):
            if hit:
                rr = 1.0 / rank
                break

        dcg = 0.0
        for rank, hit in enumerate(hit_flags, start=1):
            if hit:
                dcg += 1.0 / math.log2(rank + 1)

        ideal_hits = min(len(true_items), k)
        idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
        ndcg = 0.0 if idcg == 0 else dcg / idcg

        rows.append({
            "hit": hit_at_k,
            "recall": recall_at_k,
            "mrr": rr,
            "ndcg": ndcg,
        })

    if not rows:
        return {
            "users_eval": 0,
            "hit@k": 0.0,
            "recall@k": 0.0,
            "mrr@k": 0.0,
            "ndcg@k": 0.0,
        }

    out = pd.DataFrame(rows)
    return {
        "users_eval": len(out),
        "hit@k": out["hit"].mean(),
        "recall@k": out["recall"].mean(),
        "mrr@k": out["mrr"].mean(),
        "ndcg@k": out["ndcg"].mean(),
    }


def evaluate_model(model_name, recs, split_df, split_name="val", k_list=EVAL_KS):
    rec_map = recs if isinstance(recs, dict) else recommendation_df_to_map(recs, topk=max(k_list))
    seen_map = make_seen_map(train_reviews)

    all_truth = make_truth_map(split_df, cold_only=False)
    cold_truth = make_truth_map(split_df, cold_only=True)

    eval_rows = []
    for k in k_list:
        for group_name, truth_map in [("all_positive", all_truth), ("cold_start_positive", cold_truth)]:
            scores = evaluate_rec_map(rec_map, truth_map, seen_map=seen_map, k=k)
            eval_rows.append({
                "model": model_name,
                "split": split_name,
                "group": group_name,
                "k": k,
                **scores,
            })

    return pd.DataFrame(eval_rows)


print("Evaluator ready")
print("positive review cutoff:", MIN_POSITIVE_STARS)
print("val positive users:", len(make_truth_map(val_reviews)))
print("val cold-start positive users:", len(make_truth_map(val_reviews, cold_only=True)))
print("test positive users:", len(make_truth_map(test_reviews)))
print("test cold-start positive users:", len(make_truth_map(test_reviews, cold_only=True)))


Evaluator ready
positive review cutoff: 4.0
val positive users: 7551
val cold-start positive users: 363
test positive users: 5003
test cold-start positive users: 183


In [ ]:
# # =========================================================
# # Small evaluator smoke test: popularity baseline
# # =========================================================
# # This is not the final model. It just proves the evaluator works.

# train_pop_items = (
#     train_reviews.groupby("business_id")
#     .size()
#     .sort_values(ascending=False)
#     .index
#     .tolist()
# )

# val_users = sorted(val_reviews["user_id"].unique())
# pop_recs = {uid: train_pop_items[:100] for uid in val_users}

# pop_val_scores = evaluate_model(
#     "popularity_smoke_test",
#     pop_recs,
#     val_reviews,
#     split_name="val",
#     k_list=[5, 10, 20]
# )

# display(pop_val_scores)


,model,split,group,k,users_eval,hit@k,recall@k,mrr@k,ndcg@k
0,popularity_smoke_test,val,all_positive,5,7551,0.058138,0.024002,0.028610,0.020065
1,popularity_smoke_test,val,cold_start_positive,5,363,0.000000,0.000000,0.000000,0.000000
2,popularity_smoke_test,val,all_positive,10,7551,0.095219,0.042510,0.033555,0.026385
3,popularity_smoke_test,val,cold_start_positive,10,363,0.000000,0.000000,0.000000,0.000000
4,popularity_smoke_test,val,all_positive,20,7551,0.153887,0.075510,0.037544,0.036200
5,popularity_smoke_test,val,cold_start_positive,20,363,0.000000,0.000000,0.000000,0.000000


## Matrix Factorization Baseline - Anunay

 It follows the latent factor idea from class: build a user-restaurant interaction matrix from positive training reviews, factor it into lower-dimensional user and restaurant vectors, then recommend unseen restaurants using user-vector dot item-vector scores.


In [ ]:
# =========================================================
# Matrix factorization baseline with TruncatedSVD
# =========================================================
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

MF_FACTORS = 64
MF_TOPN = 100
RANDOM_SEED = 172
RESULTS_DIR = os.path.join(DATA_DIR, "model_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Positive review = user liked the restaurant enough to use as implicit feedback.
train_pos = train_reviews[train_reviews["stars"] >= MIN_POSITIVE_STARS].copy()

# If a user reviewed the same restaurant more than once, one positive signal is enough.
train_pos = train_pos[["user_id", "business_id"]].drop_duplicates()

mf_users = sorted(train_reviews["user_id"].unique())
mf_items = sorted(train_reviews["business_id"].unique())

user_to_row = {uid: i for i, uid in enumerate(mf_users)}
item_to_col = {bid: i for i, bid in enumerate(mf_items)}
row_to_user = {i: uid for uid, i in user_to_row.items()}
col_to_item = {i: bid for bid, i in item_to_col.items()}

row_idx = train_pos["user_id"].map(user_to_row).to_numpy()
col_idx = train_pos["business_id"].map(item_to_col).to_numpy()
data = np.ones(len(train_pos), dtype=np.float32)

user_item_mat = csr_matrix(
    (data, (row_idx, col_idx)),
    shape=(len(mf_users), len(mf_items))
)

print("matrix shape:", user_item_mat.shape)
print("positive train pairs:", user_item_mat.nnz)

mf = TruncatedSVD(n_components=MF_FACTORS, random_state=RANDOM_SEED)
user_factors = mf.fit_transform(user_item_mat)
item_factors = mf.components_.T

print("latent factors:", MF_FACTORS)
print("variance captured:", round(float(mf.explained_variance_ratio_.sum()), 4))

# Popularity fallback keeps recommendations full for users with weak/empty positive history.
train_pop_items = (
    train_reviews.groupby("business_id")
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)

seen_map = make_seen_map(train_reviews)


def fill_with_popular(recs, seen_items, topn=MF_TOPN):
    out = []
    used = set()

    for item in recs:
        if item in used or item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == topn:
            return out

    for item in train_pop_items:
        if item in used or item in seen_items:
            continue
        out.append(item)
        used.add(item)
        if len(out) == topn:
            break

    return out


def mf_recs_for_users(user_list, topn=MF_TOPN):
    rec_map = {}

    for uid in user_list:
        seen_items = seen_map.get(uid, set())

        if uid not in user_to_row:
            rec_map[uid] = fill_with_popular([], seen_items, topn=topn)
            continue

        uidx = user_to_row[uid]
        scores = user_factors[uidx] @ item_factors.T

        for item in seen_items:
            col = item_to_col.get(item)
            if col is not None:
                scores[col] = -np.inf

        take_n = min(topn * 3, len(mf_items))
        best_cols = np.argpartition(-scores, take_n - 1)[:take_n]
        best_cols = best_cols[np.argsort(-scores[best_cols])]
        raw_recs = [col_to_item[c] for c in best_cols if np.isfinite(scores[c])]

        rec_map[uid] = fill_with_popular(raw_recs, seen_items, topn=topn)

    return rec_map


val_users = sorted(val_reviews["user_id"].unique())
test_users = sorted(test_reviews["user_id"].unique())

mf_val_recs = mf_recs_for_users(val_users, topn=MF_TOPN)
mf_test_recs = mf_recs_for_users(test_users, topn=MF_TOPN)

mf_val_scores = evaluate_model(
    "matrix_factorization_svd",
    mf_val_recs,
    val_reviews,
    split_name="val",
    k_list=[5, 10, 20]
)

mf_test_scores = evaluate_model(
    "matrix_factorization_svd",
    mf_test_recs,
    test_reviews,
    split_name="test",
    k_list=[5, 10, 20]
)

mf_scores = pd.concat([mf_val_scores, mf_test_scores], ignore_index=True)
mf_scores.to_csv(os.path.join(RESULTS_DIR, "matrix_factorization_scores.csv"), index=False)

print("saved:", os.path.join(RESULTS_DIR, "matrix_factorization_scores.csv"))
display(mf_scores)


matrix shape: (124626, 2890)
positive train pairs: 244402
latent factors: 64
variance captured: 0.2836
saved: /content/yelp_philly_processed/model_results/matrix_factorization_scores.csv


,model,split,group,k,users_eval,hit@k,recall@k,mrr@k,ndcg@k
0,matrix_factorization_svd,val,all_positive,5,7551,0.041319,0.015418,0.020117,0.013954
1,matrix_factorization_svd,val,cold_start_positive,5,363,0.000000,0.000000,0.000000,0.000000
2,matrix_factorization_svd,val,all_positive,10,7551,0.069395,0.029089,0.023749,0.018203
3,matrix_factorization_svd,val,cold_start_positive,10,363,0.000000,0.000000,0.000000,0.000000
4,matrix_factorization_svd,val,all_positive,20,7551,0.110184,0.047562,0.026539,0.023688
5,matrix_factorization_svd,val,cold_start_positive,20,363,0.000000,0.000000,0.000000,0.000000
6,matrix_factorization_svd,test,all_positive,5,5003,0.032780,0.012802,0.016443,0.011422
7,matrix_factorization_svd,test,cold_start_positive,5,183,0.000000,0.000000,0.000000,0.000000
8,matrix_factorization_svd,test,all_positive,10,5003,0.055167,0.023056,0.019250,0.014647
9,matrix_factorization_svd,test,cold_start_positive,10,183,0.000000,0.000000,0.000000,0.000000


# Popularity - Aaron